# TxGuard - Wallet Anomaly Detection Model

**Isolation Forest** for detecting anomalous wallet behavior on blockchain.

This notebook is self-contained for Google Colab - no external database required.

In [ ]:
# Install dependencies
!pip install -q scikit-learn joblib numpy pandas matplotlib seaborn

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime
from typing import Dict, List, Any, Optional
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Feature Definitions (47 features)

In [ ]:
NUMERICAL_FEATURES = [
    "transaction_count",
    "incoming_count",
    "outgoing_count",
    "total_received",
    "total_sent",
    "average_transaction_amount",
    "maximum_transaction_amount",
    "transaction_frequency",
    "active_days",
    "average_time_between_transactions",
    "transaction_burst_score",
    "unique_counterparties",
    "incoming_counterparties",
    "outgoing_counterparties",
    "incoming_outgoing_ratio",
    "rapid_transfer_ratio",
    "multi_hop_connections",
    "degree",
    "in_degree",
    "out_degree",
    "clustering_coefficient",
    "connected_component_size",
]

FEATURE_DESCRIPTIONS = {
    "transaction_count": "Total number of transactions involving this wallet",
    "incoming_count": "Number of incoming transactions (wallet as recipient)",
    "outgoing_count": "Number of outgoing transactions (wallet as sender)",
    "total_received": "Total BTC received across all incoming transactions",
    "total_sent": "Total BTC sent across all outgoing transactions",
    "average_transaction_amount": "Mean transaction amount in BTC",
    "maximum_transaction_amount": "Largest single transaction amount in BTC",
    "transaction_frequency": "Average transactions per active day",
    "active_days": "Number of days between first and last transaction (min 1)",
    "average_time_between_transactions": "Mean time gap between consecutive transactions in hours",
    "transaction_burst_score": "Ratio of actual to expected transaction frequency (higher = bursty)",
    "unique_counterparties": "Total distinct wallets transacted with",
    "incoming_counterparties": "Distinct wallets that sent to this wallet",
    "outgoing_counterparties": "Distinct wallets this wallet sent to",
    "incoming_outgoing_ratio": "Total received / total sent (inf if sent=0)",
    "rapid_transfer_ratio": "Fraction of transactions labeled 'rapid_transfer'",
    "multi_hop_connections": "Counterparties that have their own connections (depth-2)",
    "degree": "Total connections in transaction graph (depth-1)",
    "in_degree": "Incoming connections in transaction graph",
    "out_degree": "Outgoing connections in transaction graph",
    "clustering_coefficient": "Local clustering coefficient from graph",
    "connected_component_size": "Size of connected component in depth-1 graph",
}

print(f"Total features: {len(NUMERICAL_FEATURES)}")
print("Features:", NUMERICAL_FEATURES)

## 2. Model Configuration (from trained model)

In [ ]:
@dataclass
class ModelConfig:
    feature_names: List[str]
    numerical_features: List[str]
    contamination: float
    n_estimators: int
    max_samples: str
    random_state: int
    trained_at: str
    n_training_samples: int
    feature_means: Dict[str, float]
    feature_stds: Dict[str, float]

# Exact config from trained model (models/wallet_anomaly_feature_config.json)
MODEL_CONFIG = ModelConfig(
    feature_names=NUMERICAL_FEATURES,
    numerical_features=NUMERICAL_FEATURES,
    contamination=0.1,
    n_estimators=200,
    max_samples="auto",
    random_state=42,
    trained_at="2026-09-10T04:32:57.876907",
    n_training_samples=330,
    feature_means={
        "transaction_count": 71.0,
        "incoming_count": 36.36363636363637,
        "outgoing_count": 36.36363636363637,
        "total_received": 256163.09131283272,
        "total_sent": 256163.09131283272,
        "average_transaction_amount": 8386.149538344607,
        "maximum_transaction_amount": 255133.75599834856,
        "transaction_frequency": 9.071158008658008,
        "active_days": 7.815151515151515,
        "average_time_between_transactions": 3.087335042715879,
        "transaction_burst_score": 1.0636015916284265,
        "unique_counterparties": 53.86666666666667,
        "incoming_counterparties": 28.66060606060606,
        "outgoing_counterparties": 28.66060606060606,
        "incoming_outgoing_ratio": 1879.5414732289623,
        "rapid_transfer_ratio": 0.06060606060606061,
        "multi_hop_connections": 6.63030303030303,
        "degree": 57.32121212121212,
        "in_degree": 28.66060606060606,
        "out_degree": 28.66060606060606,
        "clustering_coefficient": 0.0,
        "connected_component_size": 54.587878787878786
    },
    feature_stds={
        "transaction_count": 29.271914607733326,
        "incoming_count": 17.118251660965758,
        "outgoing_count": 23.534386898236317,
        "total_received": 340900.928394473,
        "total_sent": 642486.7806947933,
        "average_transaction_amount": 11399.34968508844,
        "maximum_transaction_amount": 285770.95168885717,
        "transaction_frequency": 3.6128249653098847,
        "active_days": 0.38876408651944977,
        "average_time_between_transactions": 0.7654201267996387,
        "transaction_burst_score": 0.03840497129734081,
        "unique_counterparties": 14.790136092451348,
        "incoming_counterparties": 11.614107849995666,
        "outgoing_counterparties": 15.079322975335867,
        "incoming_outgoing_ratio": 3202.2119849229553,
        "rapid_transfer_ratio": 0.23896864763567435,
        "multi_hop_connections": 8.527605266405455,
        "degree": 12.7946074691338,
        "in_degree": 11.614107849995666,
        "out_degree": 15.079322975335867,
        "clustering_coefficient": 0.0,
        "connected_component_size": 15.043397355565032
    }
)

print("Model Config:")
print(f"  Contamination: {MODEL_CONFIG.contamination}")
print(f"  N Estimators: {MODEL_CONFIG.n_estimators}")
print(f"  Training Samples: {MODEL_CONFIG.n_training_samples}")
print(f"  Trained At: {MODEL_CONFIG.trained_at}")

## 3. Create Model Pipeline (Isolation Forest + RobustScaler)

In [ ]:
def create_pipeline(contamination: float = 0.1, n_estimators: int = 200, random_state: int = 42) -> Pipeline:
    """Create Isolation Forest pipeline with RobustScaler."""
    pipeline = Pipeline([
        ("scaler", RobustScaler()),
        ("model", IsolationForest(
            contamination=contamination,
            n_estimators=n_estimators,
            max_samples="auto",
            random_state=random_state,
            n_jobs=-1,
            verbose=0,
        )),
    ])
    return pipeline

# Create and inspect pipeline
pipeline = create_pipeline(
    contamination=MODEL_CONFIG.contamination,
    n_estimators=MODEL_CONFIG.n_estimators,
    random_state=MODEL_CONFIG.random_state
)
print("Pipeline steps:", [step[0] for step in pipeline.steps])

## 4. Generate Synthetic Training Data (for demo)

In [ ]:
def generate_synthetic_wallets(n_normal: int = 300, n_anomalous: int = 30, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic wallet features for demonstration."""
    np.random.seed(seed)
    
    # Normal wallets - clustered around feature means
    normal_data = {}
    for feat in NUMERICAL_FEATURES:
        mean = MODEL_CONFIG.feature_means[feat]
        std = MODEL_CONFIG.feature_stds[feat]
        if std == 0:
            normal_data[feat] = np.full(n_normal, mean)
        else:
            # Use log-normal for heavily skewed features
            if feat in ["total_received", "total_sent", "maximum_transaction_amount", "incoming_outgoing_ratio"]:
                # Log-normal for monetary values
                log_mean = np.log(max(mean, 1))
                log_std = std / max(mean, 1)
                normal_data[feat] = np.random.lognormal(log_mean, log_std, n_normal)
            else:
                normal_data[feat] = np.random.normal(mean, std, n_normal)
    
    normal_df = pd.DataFrame(normal_data)
    normal_df = normal_df.clip(lower=0)  # No negative values
    
    # Anomalous wallets - outliers in multiple dimensions
    anomalous_data = {}
    for feat in NUMERICAL_FEATURES:
        mean = MODEL_CONFIG.feature_means[feat]
        std = MODEL_CONFIG.feature_stds[feat]
        if std == 0:
            anomalous_data[feat] = np.full(n_anomalous, mean)
        else:
            # Generate outliers (3-5 std deviations away)
            direction = np.random.choice([-1, 1], n_anomalous)
            magnitude = np.random.uniform(3, 5, n_anomalous)
            if feat in ["total_received", "total_sent", "maximum_transaction_amount", "incoming_outgoing_ratio"]:
                log_mean = np.log(max(mean, 1))
                log_std = std / max(mean, 1)
                base = np.random.lognormal(log_mean, log_std, n_anomalous)
                anomalous_data[feat] = base * np.exp(direction * magnitude * log_std)
            else:
                anomalous_data[feat] = mean + direction * magnitude * std
    
    anomalous_df = pd.DataFrame(anomalous_data)
    anomalous_df = anomalous_df.clip(lower=0)
    
    # Combine
    df = pd.concat([normal_df, anomalous_df], ignore_index=True)
    labels = np.array([0] * n_normal + [1] * n_anomalous)  # 0=normal, 1=anomalous
    
    return df, labels

# Generate data
df_train, y_true = generate_synthetic_wallets(300, 30)
print(f"Training data shape: {df_train.shape}")
print(f"True anomalies: {y_true.sum()}")
print(df_train.describe())

## 5. Train Model on Synthetic Data

In [ ]:
# Train the model
print("Training Isolation Forest...")
pipeline.fit(df_train)

# Get predictions
scores = pipeline.decision_function(df_train)
predictions = pipeline.predict(df_train)  # 1=normal, -1=anomaly
anomaly_scores = -scores  # Higher = more anomalous

n_anomalies = sum(p == -1 for p in predictions)
anomaly_rate = n_anomalies / len(predictions)

print(f"Training complete:")
print(f"  Total samples: {len(df_train)}")
print(f"  Anomalies detected: {n_anomalies} ({anomaly_rate:.2%})")
print(f"  Score range: [{anomaly_scores.min():.4f}, {anomaly_scores.max():.4f}]")

# Compare with ground truth
from sklearn.metrics import classification_report
y_pred_binary = (predictions == -1).astype(int)
print("\nClassification Report:")
print(classification_report(y_true, y_pred_binary, target_names=["Normal", "Anomalous"]))

## 6. Visualize Anomaly Scores

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Anomaly score distribution
axes[0, 0].hist(anomaly_scores[y_true==0], bins=30, alpha=0.7, label='Normal', density=True)
axes[0, 0].hist(anomaly_scores[y_true==1], bins=30, alpha=0.7, label='Anomalous', density=True)
axes[0, 0].axvline(pipeline.named_steps["model"].offset_, color='red', linestyle='--', label='Threshold')
axes[0, 0].set_xlabel('Anomaly Score')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Anomaly Score Distribution')
axes[0, 0].legend()

# 2. Score vs index
colors = ['red' if a == -1 else 'blue' for a in predictions]
axes[0, 1].scatter(range(len(anomaly_scores)), anomaly_scores, c=colors, alpha=0.6, s=20)
axes[0, 1].axhline(pipeline.named_steps["model"].offset_, color='red', linestyle='--', label='Threshold')
axes[0, 1].set_xlabel('Sample Index')
axes[0, 1].set_ylabel('Anomaly Score')
axes[0, 1].set_title('Anomaly Scores by Sample')
axes[0, 1].legend()

# 3. Feature importance (using permutation on a few features)
from sklearn.inspection import permutation_importance
result = permutation_importance(pipeline, df_train, predictions, n_repeats=5, random_state=42, n_jobs=-1)
importances = pd.Series(result.importances_mean, index=NUMERICAL_FEATURES).sort_values(ascending=False)
top_10 = importances.head(10)
top_10.plot(kind='barh', ax=axes[1, 0])
axes[1, 0].set_title('Top 10 Feature Importances (Permutation)')
axes[1, 0].invert_yaxis()

# 4. 2D projection (first 2 features)
scatter = axes[1, 1].scatter(df_train.iloc[:, 0], df_train.iloc[:, 1], c=anomaly_scores, cmap='RdYlBu_r', alpha=0.6)
axes[1, 1].set_xlabel(NUMERICAL_FEATURES[0])
axes[1, 1].set_ylabel(NUMERICAL_FEATURES[1])
axes[1, 1].set_title('Feature Space (First 2 Features)')
plt.colorbar(scatter, ax=axes[1, 1], label='Anomaly Score')

plt.tight_layout()
plt.show()

## 7. Save Model (for later use in Colab)

In [ ]:
# Save model and config
MODEL_DIR = Path("/content/txguard_models")
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "wallet_anomaly_model.joblib"
CONFIG_PATH = MODEL_DIR / "wallet_anomaly_feature_config.json"

joblib.dump(pipeline, MODEL_PATH)
with open(CONFIG_PATH, "w") as f:
    json.dump(asdict(MODEL_CONFIG), f, indent=2)

print(f"Model saved to: {MODEL_PATH}")
print(f"Config saved to: {CONFIG_PATH}")
print(f"Model size: {MODEL_PATH.stat().st_size / 1024:.1f} KB")

## 8. Load Model & Predict on New Wallets

In [ ]:
def load_model(model_path: Path, config_path: Path) -> tuple:
    """Load trained pipeline and config."""
    pipeline = joblib.load(model_path)
    with open(config_path, "r") as f:
        config_dict = json.load(f)
    config = ModelConfig(**config_dict)
    return pipeline, config

# Reload
loaded_pipeline, loaded_config = load_model(MODEL_PATH, CONFIG_PATH)
print(f"Loaded model trained on: {loaded_config.trained_at}")
print(f"Training samples: {loaded_config.n_training_samples}")

In [ ]:
@dataclass
class AnomalyResult:
    wallet_address: str
    is_anomaly: bool
    anomaly_score: float
    threshold: float
    model_version: str
    model_trained_at: str
    features_used: Dict[str, float]
    feature_descriptions: Dict[str, str]

def predict_anomaly(wallet_address: str, features: Dict[str, float], pipeline: Pipeline, config: ModelConfig) -> AnomalyResult:
    """Predict anomaly for a single wallet given its features."""
    # Build feature vector in correct order
    feature_vector = []
    features_used = {}
    for feat_name in config.feature_names:
        value = features.get(feat_name, 0.0)
        if value is None:
            value = 0.0
        if isinstance(value, float) and (np.isnan(value) or np.isinf(value)):
            value = 0.0
        feature_vector.append(float(value))
        features_used[feat_name] = float(value)

    X = np.array(feature_vector).reshape(1, -1)

    score = pipeline.decision_function(X)[0]
    prediction = pipeline.predict(X)[0]

    anomaly_score = -score
    is_anomaly = prediction == -1
    threshold = -pipeline.named_steps["model"].offset_ if hasattr(pipeline.named_steps["model"], "offset_") else 0.0

    return AnomalyResult(
        wallet_address=wallet_address,
        is_anomaly=bool(is_anomaly),
        anomaly_score=float(anomaly_score),
        threshold=float(threshold),
        model_version=f"isolation_forest_v1_{config.random_state}",
        model_trained_at=config.trained_at,
        features_used=features_used,
        feature_descriptions={k: v for k, v in FEATURE_DESCRIPTIONS.items() if k in config.feature_names},
    )

## 9. Example Predictions

In [ ]:
# Example 1: Normal wallet (near training mean)
normal_wallet = {
    "transaction_count": 70,
    "incoming_count": 35,
    "outgoing_count": 35,
    "total_received": 250000,
    "total_sent": 250000,
    "average_transaction_amount": 8000,
    "maximum_transaction_amount": 200000,
    "transaction_frequency": 9.0,
    "active_days": 8,
    "average_time_between_transactions": 3.0,
    "transaction_burst_score": 1.05,
    "unique_counterparties": 50,
    "incoming_counterparties": 28,
    "outgoing_counterparties": 28,
    "incoming_outgoing_ratio": 1.0,
    "rapid_transfer_ratio": 0.05,
    "multi_hop_connections": 7,
    "degree": 55,
    "in_degree": 28,
    "out_degree": 28,
    "clustering_coefficient": 0.0,
    "connected_component_size": 55
}

result1 = predict_anomaly("1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa", normal_wallet, loaded_pipeline, loaded_config)
print("=== Normal Wallet ===")
print(f"Address: {result1.wallet_address}")
print(f"Is Anomaly: {result1.is_anomaly}")
print(f"Anomaly Score: {result1.anomaly_score:.4f}")
print(f"Threshold: {result1.threshold:.4f}")

# Example 2: Suspicious wallet (high velocity, many counterparties)
suspicious_wallet = {
    "transaction_count": 500,
    "incoming_count": 250,
    "outgoing_count": 250,
    "total_received": 5000000,
    "total_sent": 4900000,
    "average_transaction_amount": 15000,
    "maximum_transaction_amount": 1000000,
    "transaction_frequency": 50,
    "active_days": 10,
    "average_time_between_transactions": 0.5,
    "transaction_burst_score": 5.0,
    "unique_counterparties": 300,
    "incoming_counterparties": 150,
    "outgoing_counterparties": 150,
    "incoming_outgoing_ratio": 1.02,
    "rapid_transfer_ratio": 0.8,
    "multi_hop_connections": 50,
    "degree": 200,
    "in_degree": 150,
    "out_degree": 150,
    "clustering_coefficient": 0.1,
    "connected_component_size": 200
}

result2 = predict_anomaly("1F1tAaz5x1HUXrCNLbtMDqcw6o5GNn4xqX", suspicious_wallet, loaded_pipeline, loaded_config)
print("\n=== Suspicious Wallet ===")
print(f"Address: {result2.wallet_address}")
print(f"Is Anomaly: {result2.is_anomaly}")
print(f"Anomaly Score: {result2.anomaly_score:.4f}")
print(f"Threshold: {result2.threshold:.4f}")

# Example 3: Dusting attack pattern (many tiny incoming)
dusting_wallet = {
    "transaction_count": 1000,
    "incoming_count": 950,
    "outgoing_count": 50,
    "total_received": 1000,
    "total_sent": 50000,
    "average_transaction_amount": 1.0,
    "maximum_transaction_amount": 100,
    "transaction_frequency": 100,
    "active_days": 10,
    "average_time_between_transactions": 0.1,
    "transaction_burst_score": 10.0,
    "unique_counterparties": 800,
    "incoming_counterparties": 800,
    "outgoing_counterparties": 50,
    "incoming_outgoing_ratio": 0.02,
    "rapid_transfer_ratio": 0.9,
    "multi_hop_connections": 20,
    "degree": 800,
    "in_degree": 800,
    "out_degree": 50,
    "clustering_coefficient": 0.0,
    "connected_component_size": 800
}

result3 = predict_anomaly("1DustyWallet...", dusting_wallet, loaded_pipeline, loaded_config)
print("\n=== Dusting Attack Wallet ===")
print(f"Address: {result3.wallet_address}")
print(f"Is Anomaly: {result3.is_anomaly}")
print(f"Anomaly Score: {result3.anomaly_score:.4f}")
print(f"Threshold: {result3.threshold:.4f}")

## 10. Batch Prediction on Multiple Wallets

In [ ]:
def predict_batch(wallet_features: List[Dict[str, Any]], pipeline: Pipeline, config: ModelConfig) -> List[AnomalyResult]:
    """Predict anomalies for multiple wallets."""
    results = []
    for wallet in wallet_features:
        addr = wallet.pop("wallet_address", "unknown")
        result = predict_anomaly(addr, wallet, pipeline, config)
        results.append(result)
    return results

# Test batch
test_wallets = [
    {"wallet_address": "wallet_1", **normal_wallet},
    {"wallet_address": "wallet_2", **suspicious_wallet},
    {"wallet_address": "wallet_3", **dusting_wallet},
    # Add some variations
    {"wallet_address": "wallet_4", **{k: v*1.2 for k, v in normal_wallet.items()}},
    {"wallet_address": "wallet_5", **{k: v*0.8 for k, v in normal_wallet.items()}},
]

batch_results = predict_batch(test_wallets, loaded_pipeline, loaded_config)

print(f"{'Address':<20} {'Anomaly':<10} {'Score':<10} {'Threshold':<10}")
print("-" * 50)
for r in batch_results:
    print(f"{r.wallet_address:<20} {str(r.is_anomaly):<10} {r.anomaly_score:<10.4f} {r.threshold:<10.4f}")

## 11. Export Results to JSON (for API integration)

In [ ]:
import json
from dataclasses import asdict

# Convert results to JSON-serializable format
def result_to_dict(result: AnomalyResult) -> Dict:
    return asdict(result)

output = {
    "model_info": {
        "version": f"isolation_forest_v1_{loaded_config.random_state}",
        "trained_at": loaded_config.trained_at,
        "training_samples": loaded_config.n_training_samples,
        "contamination": loaded_config.contamination,
        "n_estimators": loaded_config.n_estimators,
    },
    "predictions": [result_to_dict(r) for r in batch_results]
}

with open("/content/txguard_predictions.json", "w") as f:
    json.dump(output, f, indent=2)

print("Predictions saved to /content/txguard_predictions.json")
print(json.dumps(output, indent=2))

## 12. Download Model Files (for local use)

In [ ]:
from google.colab import files

# Download model files
files.download(str(MODEL_PATH))
files.download(str(CONFIG_PATH))